# 시계열 조기경보 파이프라인 — 실데이터로 따라가기

PhysioNet Challenge 2012 (ICU 사망 예측, 4,000명) 으로 이 프로젝트의 파이프라인을
처음부터 끝까지 실행하며 이해합니다.

## 02번 노트북과 무엇이 다른가

`02_learning_project.ipynb`는 한 행 = 한 사람인 표 데이터였습니다. 이 노트북은
한 사람이 여러 시점의 기록을 갖는 시계열이라, 새로운 개념이 여럿 등장합니다.

| | 02번 (표 데이터) | 이 노트북 (시계열) |
|---|---|---|
| 데이터 단위 | 환자 1명 = 1행 | 환자 1명 = 수십~수백 행 |
| 예측 대상 | 이 사람이 당뇨인가 | 지금부터 N시간 안에 사건이 오는가 |
| 학습 단위 | 환자 | 관찰 윈도우 (한 환자에서 여러 개) |
| 분할 | 무작위 | 환자 단위 (안 그러면 누수) |
| 새 개념 | — | 슬라이딩 윈도우 · 예측 지평 · lead-time · 알람 부담 |

## 오늘 배울 것

| 절 | 내용 | 왜 중요한가 |
|:--:|---|---|
| 1–2 | 데이터 로드와 코호트 구조 | 시계열 데이터가 어떻게 생겼는지 |
| 3 | EDA — 분포·결측·궤적 | 사건 전에 활력징후가 실제로 변하는지 눈으로 |
| 4 | 슬라이딩 윈도우와 예측 지평 | 시계열을 학습 가능한 형태로 바꾸는 핵심 |
| 5 | 환자 단위 분할 | 시계열에서 가장 흔한 치명적 실수 |
| 6 | 개인 기저선 이탈 피처 | 이 프로젝트의 차별점 |
| 7 | XGBoost vs NEWS | 임상 점수를 이기는가 |
| 8 | 알람 부담과 lead-time | 이 프로젝트가 진짜 보는 지표 |
| 9 | SHAP | 왜 위험한지 설명 |

> 데이터 성격: Challenge 2012의 사건은 원내 사망이지 심정지가 아닙니다. 또
> 사망 시각이 일 단위로만 기록돼 있어 거칩니다. 여기 숫자는 "파이프라인이 실데이터에서
> 작동한다"는 근거이지, 경북대 심정지 성능이 아닙니다.

> 플롯 제목은 영어입니다 (서버에 한글 폰트가 없으면 □로 깨짐). 설명은 마크다운에 한글로.

## 0. 설정

`DATA_DIR`와 `OUTCOMES`를 서버의 실제 경로로 맞추세요.
경로가 없으면 문서 형식대로 만든 예시 데이터를 자동 생성해서 노트북이 끝까지 돌아갑니다
(그 경우 화면에 크게 표시됩니다).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# ============ 설정 (서버 경로에 맞게 수정) ============
DATA_DIR  = "/workspace/set-a"                 # Challenge 2012 환자 .txt 폴더
OUTCOMES  = "/workspace/Outcomes-a.txt"        # 라벨 파일
MAX_FILES = 800        # 읽을 환자 수 (None=전체 4000, 느려짐)
HORIZON   = 6          # 예측 지평(시간) — 4절에서 이 값의 의미를 배웁니다
WINDOW    = 8          # 관찰 윈도우 길이(시간)
USE_GPU   = False
# ====================================================

VITALS = ["pulse", "sbp", "dbp", "temperature", "spo2", "resp_rate"]
print("repo:", REPO)

In [ ]:
# 데이터가 없으면 문서 형식대로 예시 코호트를 만들어 노트북이 계속 돌게 한다
USING_DEMO_DATA = not Path(DATA_DIR).is_dir()

if USING_DEMO_DATA:
    print("=" * 78)
    print(" 실제 Challenge-2012 데이터를 찾지 못해 예시 데이터를 생성합니다.")
    print(f"   찾은 경로: {DATA_DIR}")
    print("   아래 수치는 전부 인위적으로 만든 신호이므로 성능으로 읽으면 안 됩니다.")
    print("   실데이터 받는 법은 이 노트북 맨 아래를 참고하세요.")
    print("=" * 78)

    demo = REPO / "models" / "_demo_challenge2012"
    demo.mkdir(parents=True, exist_ok=True)
    for old in demo.glob("*.txt"):
        old.unlink()

    rng = np.random.default_rng(42)
    outcome_rows = ["RecordID,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death"]
    for i in range(300):
        rid = 100000 + i
        dies = i % 8 == 0                       # 약 12.5%가 하루 안에 사망
        lines = ["Time,Parameter,Value", f"00:00,RecordID,{rid}",
                 f"00:00,Age,{rng.integers(45, 90)}", f"00:00,Gender,{i % 2}",
                 "00:00,Height,-1", "00:00,ICUType,4"]
        for h in range(48):
            drift = (h / 48) * (25 if dies else 0)   # 악화 환자만 서서히 나빠짐
            m = lambda: rng.integers(0, 59)
            lines += [
                f"{h:02d}:{m():02d},HR,{80 + drift + rng.normal(0, 4):.1f}",
                f"{h:02d}:{m():02d},SysABP,{120 - drift + rng.normal(0, 5):.1f}",
                f"{h:02d}:{m():02d},DiasABP,{70 + rng.normal(0, 4):.1f}",
                f"{h:02d}:{m():02d},Temp,{37 + rng.normal(0, 0.3):.1f}",
                f"{h:02d}:{m():02d},SaO2,{98 - drift / 5 + rng.normal(0, 1):.1f}",
                f"{h:02d}:{m():02d},RespRate,{16 + drift / 4 + rng.normal(0, 2):.1f}",
            ]
        (demo / f"{rid}.txt").write_text("\n".join(lines) + "\n")
        outcome_rows.append(f"{rid},6,1,5,{1 if dies else -1},{1 if dies else 0}")

    (demo / "Outcomes-a.txt").write_text("\n".join(outcome_rows) + "\n")
    DATA_DIR, OUTCOMES, MAX_FILES = str(demo), str(demo / "Outcomes-a.txt"), None
    print(f"\n예시 데이터 300명 생성 완료 → {demo}")
else:
    print(f"실데이터 사용: {DATA_DIR}")

## 1. 데이터 로드

### 원본이 어떻게 생겼나

Challenge 2012의 각 환자 파일은 긴 형태(long format) 입니다. 한 줄에 하나의 측정값이
`시각, 항목, 값`으로 들어 있어요.

```
Time,Parameter,Value
00:00,RecordID,132539     ← 환자 번호
00:00,Age,54              ← 나이 (시각 00:00에 기록된 "정적" 정보)
00:00,Height,-1           ← -1 = 측정 안 함 (결측!)
00:07,HR,73               ← 여기부터 시계열
00:07,SysABP,118
00:37,HR,77               ← 같은 시간대에 또 측정
```

이걸 그대로는 못 씁니다. 환자 × 시간 격자로 바꿔야 해요. 그 일을
`cohort_from_challenge2012()`가 합니다:

1. `시각,항목,값` → 시간별 표로 재배열
2. 같은 시간대 중복 측정 → 평균
3. 동맥압(`SysABP`) 우선, 커프(`NISysABP`)는 빈 곳만 채움
4. `-1`을 결측으로 처리 ← 안 하면 모델이 "키 -1인 환자"를 믿습니다
5. `Outcomes` 파일에서 사망 시각을 붙임

In [ ]:
from vitals_data import cohort_from_challenge2012

cohort = cohort_from_challenge2012(DATA_DIR, outcomes_path=OUTCOMES, max_files=MAX_FILES)

n_patients = cohort.vitals["patient_id"].nunique()
n_event = int(cohort.events["arrest_hour"].notna().sum())
print(f"환자 수      : {n_patients}")
print(f"사건(사망) 환자: {n_event}  ({n_event / n_patients:.1%})")
print(f"활력징후 행 수 : {len(cohort.vitals):,}")
print(f"환자당 평균    : {len(cohort.vitals) / n_patients:.1f}행")

### 코호트는 3개의 표로 되어 있습니다

파이프라인의 모든 데이터(합성·MIMIC·경북대·Challenge)는 똑같은 이 3개 표로 변환됩니다.
그래서 어떤 데이터든 같은 코드가 돕니다.

In [ ]:
print("① vitals — 환자 × 시간별 활력징후")
display(cohort.vitals.head())

print("\n② events — 환자별 사건 시각 (NaN = 사건 없음 = 대조군)")
display(cohort.events.head())

print("\n③ demographics — 정적 정보")
display(cohort.demographics.head())

## 2. EDA — 데이터를 눈으로 확인

먼저 결측을 봅니다. 실제 임상 데이터는 항목마다 측정 빈도가 크게 다릅니다.

In [ ]:
miss = cohort.vitals[VITALS].isna().mean().sort_values()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
(miss * 100).plot.barh(ax=ax[0], color="indianred")
ax[0].set_title("Missing rate per vital")
ax[0].set_xlabel("% missing")
for i, v in enumerate(miss.values * 100):
    ax[0].text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=9)

lengths = cohort.vitals.groupby("patient_id")["hour"].max() + 1
ax[1].hist(lengths, bins=30, color="slateblue")
ax[1].set_title("Recorded hours per patient")
ax[1].set_xlabel("hours"); ax[1].set_ylabel("patients")
plt.tight_layout(); plt.show()

print(miss.mul(100).round(1).to_string())

결측률이 항목마다 다른 것 자체가 정보입니다. 자주 재는 항목(맥박)과 드물게 재는 항목이
갈리는데, "이 환자는 왜 이 검사를 자주 했나?"에 중증도가 반영돼 있을 수 있어요.

이제 사건이 실제로 활력징후에 나타나는지 확인합니다. 이게 안 보이면 아무리 좋은 모델도
소용없습니다.

In [ ]:
# 사건 환자 한 명의 궤적
event_ids = cohort.events.loc[cohort.events["arrest_hour"].notna(), "patient_id"].tolist()
example = event_ids[0]
g = cohort.vitals[cohort.vitals.patient_id == example].sort_values("hour")
event_h = float(cohort.events.set_index("patient_id").loc[example, "arrest_hour"])

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, v in zip(axes.ravel(), VITALS):
    ax.plot(g["hour"], g[v], marker=".", ms=3, lw=1)
    if event_h <= g["hour"].max():
        ax.axvline(event_h, color="red", ls="--", lw=2)
    ax.set_title(v, fontsize=10); ax.set_xlabel("hour")
fig.suptitle(f"Patient {example} — trajectory (red = event time)", fontsize=13)
plt.tight_layout(); plt.show()

print(f"이 환자의 사건 시각: 입원 후 {event_h:.0f}시간")
print(f"기록된 마지막 시각 : {g['hour'].max():.0f}시간")

In [ ]:
# 사건군 vs 대조군의 평균 궤적 (개별 환자는 노이즈가 크므로 평균으로)
ev = set(event_ids)
tagged = cohort.vitals.assign(group=np.where(cohort.vitals.patient_id.isin(ev), "event", "control"))
profile = tagged[tagged.hour <= 48].groupby(["group", "hour"])[VITALS].mean().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, v in zip(axes.ravel(), VITALS):
    for grp, color in [("control", "steelblue"), ("event", "crimson")]:
        sub = profile[profile.group == grp]
        ax.plot(sub["hour"], sub[v], label=grp, color=color, lw=1.8)
    ax.set_title(v, fontsize=10); ax.set_xlabel("hour")
axes[0, 0].legend(fontsize=9)
fig.suptitle("Mean trajectory: event vs control patients", fontsize=13)
plt.tight_layout(); plt.show()

빨강(사건군)과 파랑(대조군) 선이 벌어지면 활력징후에 신호가 있다는 뜻입니다.
겹쳐 있으면 이 데이터로는 예측이 어렵다는 신호이고, 그럴 땐 모델을 바꿔도 소용없습니다.

> EDA에서 신호를 확인하고 모델링으로 넘어가는 이 순서가 중요합니다.
> 신호가 없는데 모델부터 돌리면 며칠을 낭비합니다.

## 3. 슬라이딩 윈도우 — 시계열을 학습 가능한 형태로

여기가 이 노트북의 핵심입니다.

### 문제

모델은 "환자 1명 = 여러 시점"을 그대로 못 먹습니다. 환자마다 길이도 다르고요.

### 해결: 시간을 잘라서 각 조각을 하나의 학습 샘플로

```
환자 A의 시간축 ──────────────────────────────▶  사건!
              [--윈도우--]                          시각 T
                    ↑ 이 8시간을 요약해서 1개 샘플로
                      라벨: "지금부터 6시간 안에 사건이 오는가?"

              [--윈도우--]        ← 1시간 뒤로 밀어서 또 하나
                    [--윈도우--]  ← 또 하나 ...
```

- 관찰 윈도우(WINDOW=8h): 과거 몇 시간을 볼 것인가
- 예측 지평(HORIZON=6h): 앞으로 몇 시간 안의 사건을 맞힐 것인가

각 윈도우에서 vital마다 평균·표준편차·최소·최대·최근값·기울기·변화량을 뽑습니다.
기울기(slope)가 중요합니다 — "지금 값"보다 "어느 방향으로 얼마나 빨리 변하는가"가
악화의 신호이기 때문입니다.

In [ ]:
from vitals_data import build_windows

windowed = build_windows(cohort, observation_window_hours=WINDOW, prediction_horizon_hours=HORIZON)

print(f"윈도우 개수  : {len(windowed.labels):,}")
print(f"양성 윈도우  : {int(windowed.labels.sum()):,}  ({windowed.labels.mean():.2%})")
print(f"피처 개수    : {windowed.features.shape[1]}")
print(f"\n피처 이름 예시: {list(windowed.features.columns[:8])}")
windowed.features.head(3).iloc[:, :8]

### 예측 지평을 바꾸면 무슨 일이 일어나나

지평이 좁을수록 양성 윈도우가 급격히 줄어듭니다. 환자당 사건은 1개뿐인데, 그 사건이
"지평 안에 들어오는" 윈도우만 양성이 되기 때문입니다.

이게 실제로 이 프로젝트에서 문제가 됐습니다. Challenge 2019를 기본값(지평 1시간)으로 돌렸더니
양성이 0.2% 밖에 안 돼서 AUPRC가 기준선까지 무너졌어요. 직접 확인해봅시다.

In [ ]:
rows = []
for h in [1, 2, 3, 6, 12, 24]:
    w = build_windows(cohort, observation_window_hours=WINDOW, prediction_horizon_hours=h)
    rows.append({"horizon_h": h, "windows": len(w.labels),
                 "positives": int(w.labels.sum()), "positive_rate": float(w.labels.mean())})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tab.horizon_h, tab.positive_rate * 100, marker="o", lw=2)
ax.axvline(HORIZON, color="green", ls=":", label=f"our choice = {HORIZON}h")
ax.set_xlabel("prediction horizon (hours)"); ax.set_ylabel("positive window rate (%)")
ax.set_title("Narrow horizon -> almost no positives -> AUPRC collapses")
for _, r in tab.iterrows():
    ax.annotate(f"{r.positive_rate*100:.2f}%", (r.horizon_h, r.positive_rate*100),
                textcoords="offset points", xytext=(0, 7), fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()

지평 선택은 통계가 아니라 임상 판단입니다.

- 너무 좁으면 → 양성이 없어서 학습 자체가 안 됨
- 너무 넓으면 → "24시간 내 사망"은 맞혀도 언제 개입해야 할지 알려주지 못함

조기경보에서는 의료진이 실제로 대응할 수 있는 시간(수 시간)이 기준이 됩니다.
NEWS 같은 임상 점수도 이 시간 규모를 가정합니다.

## 4. 환자 단위 분할 — 시계열에서 가장 흔한 치명적 실수

02번에서 "전체 평균으로 결측을 채우면 누수"라고 배웠습니다. 시계열에는 더 심한 누수가 있어요.

### 왜 무작위 분할이 안 되는가

한 환자에서 윈도우가 수십 개 나옵니다. 무작위로 나누면:

```
환자 A의 윈도우들 →  일부는 train, 일부는 test  
```

같은 환자의 hour 10 윈도우와 hour 11 윈도우는 거의 같은 데이터입니다. 하나가 train에,
다른 하나가 test에 있으면 모델은 답을 이미 본 겁니다. 성능이 크게 부풀려집니다.

### 올바른 방법: 환자 통째로

```
환자 A, C, D → train      환자 B, E → test     
```

`patient_level_split()`이 이걸 합니다. 얼마나 차이 나는지 직접 확인해봅시다.

In [ ]:
from vitals_data import add_personalized_features, patient_level_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

feat = add_personalized_features(windowed, cohort)

def fit_score(Xtr, ytr, Xte, yte):
    m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      random_state=42, n_jobs=1, eval_metric="logloss",
                      scale_pos_weight=float((ytr == 0).sum() / max((ytr == 1).sum(), 1)))
    m.fit(Xtr.fillna(Xtr.median()), ytr)
    return average_precision_score(yte, m.predict_proba(Xte.fillna(Xtr.median()))[:, 1])

# 잘못된 방법: 윈도우를 무작위로
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    feat.features, feat.labels, test_size=0.2, stratify=feat.labels, random_state=42)
auprc_wrong = fit_score(Xtr_w, ytr_w, Xte_w, yte_w)

# 올바른 방법: 환자 단위로
split = patient_level_split(feat)
auprc_right = fit_score(split.X_train, split.y_train, split.X_test, split.y_test)

base = float(split.y_test.mean())
print(f"AUPRC 기준선(양성 비율)      : {base:.4f}\n")
print(f"윈도우 무작위 분할 (누수!) : {auprc_wrong:.4f}  ({auprc_wrong/base:.1f}x)")
print(f"환자 단위 분할 (정직)      : {auprc_right:.4f}  ({auprc_right/base:.1f}x)")
print(f"\n부풀려진 정도: {auprc_wrong - auprc_right:+.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["random split\n(LEAKY)", "patient-level split\n(honest)"],
              [auprc_wrong, auprc_right], color=["indianred", "seagreen"])
ax.axhline(base, color="gray", ls="--", label=f"baseline {base:.3f}")
ax.set_ylabel("Test AUPRC"); ax.set_title("Leakage inflates the score")
for b, v in zip(bars, [auprc_wrong, auprc_right]):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom")
ax.legend(); plt.tight_layout(); plt.show()

차이가 클수록 누수가 심했다는 뜻입니다.

논문이나 대회에서 비현실적으로 높은 성능을 보면 이 지점을 의심해보세요.
시계열 의료 데이터에서 매우 흔한 실수입니다.

## 5. 개인 기저선 이탈 — 이 프로젝트의 차별점

### 아이디어

NEWS 같은 규칙 점수는 모든 환자에게 같은 임계값을 씁니다 (예: 맥박 > 90이면 위험).
그런데 평소 맥باق 55인 사람의 맥박 85와, 평소 88인 사람의 맥박 85는 전혀 다른 의미입니다.

> "환자 자신의 초기 안정기"를 기준으로, 거기서 얼마나 벗어났는가를 봅니다.

`add_personalized_features()`가 vital마다 `_last_dev`(최근값의 기저선 대비 편차)와
`_mean_dev`를 추가합니다.

### 왜 이 프로젝트에 특히 잘 맞나

경북대 데이터는 심정지 환자만 있고 대조군이 없습니다. 보통은 치명적 약점이지만,
"환자가 곧 자신의 대조군"인 이 설계에서는 오히려 자연스럽습니다.
약점을 방법론으로 바꾼 셈입니다.

실제로 도움이 되는지 확인해봅시다.

In [ ]:
# 개인 기저선 피처 유무 비교
plain = patient_level_split(windowed)          # 기본 윈도우 피처만
auprc_plain = fit_score(plain.X_train, plain.y_train, plain.X_test, plain.y_test)

n_extra = feat.features.shape[1] - windowed.features.shape[1]
print(f"기본 피처       : {windowed.features.shape[1]}개 -> AUPRC {auprc_plain:.4f} ({auprc_plain/base:.1f}x)")
print(f"+ 개인 기저선({n_extra}개): {feat.features.shape[1]}개 -> AUPRC {auprc_right:.4f} ({auprc_right/base:.1f}x)")
print(f"\n변화: {auprc_right - auprc_plain:+.4f}")

added = [c for c in feat.features.columns if c not in windowed.features.columns]
print(f"\n추가된 피처 예시: {added[:6]}")

### 위 숫자가 개선이 아니라면 (그럴 수 있습니다)

02번에서 배웠듯 피처 추가가 항상 도움이 되는 건 아닙니다. 여기서 개인 기저선이 힘을
못 쓸 수 있는 이유가 여럿 있어요:

1. 사건이 심정지가 아니라 사망 — 개인 기저선은 급성 악화를 잡도록 설계됐는데,
   사망은 더 서서히·다양한 경로로 옵니다.
2. 사건 시각이 일 단위로 거칢 — `Survival`이 정수 일수라 실제 악화 시점과 어긋납니다.
3. 트리 모델은 이미 개인차를 부분적으로 학습 — 기저선 대비 편차를 명시적으로 넣어도
   새 정보가 아닐 수 있습니다.
4. 표본이 적으면 피처가 늘수록 손해 — 예시 데이터 300명에서는 특히 그렇습니다.

할 일은 "그래도 좋다"고 우기는 게 아니라 조건을 바꿔 재확인하는 것입니다:
`MAX_FILES=None`으로 4,000명 전체, `HORIZON`을 3/12로, `PERSONAL_BASELINE_HOURS` 조정.

> 이건 정직성 문제이기도 합니다. 제안서에 "개인 기저선이 AUPRC를 올린다"고 쓰려면
> 그 주장이 성립하는 조건을 명시해야 합니다. 합성 데이터에서 올랐다고 모든 데이터에서
> 오르는 것은 아니니까요.

## 6. XGBoost vs NEWS — 임상 점수를 이기는가

NEWS(National Early Warning Score) 는 실제 병동에서 쓰는 규칙 기반 점수입니다.
각 vital을 구간별로 0~3점 매겨서 합산해요.

우리 모델은 이걸 이겨야 의미가 있습니다. 아무리 AUPRC가 높아도 NEWS와 비슷하면
"복잡한 걸 새로 도입할 이유"가 없으니까요.

In [ ]:
from vitals_train import train_xgboost, evaluate_news_baseline, compute_news_scores

model, xgb_m = train_xgboost(split, use_gpu=USE_GPU)
news_m = evaluate_news_baseline(split)

res = pd.DataFrame([{
    "model": m.model_name,
    "AUPRC": round(m.auprc, 4),
    "vs baseline": f"{m.auprc / base:.1f}x",
    "ROC-AUC": round(m.roc_auc, 3),
    "sens@95spec": round(m.sensitivity_at_95_specificity, 3),
    "falseAlarm": round(m.false_alarm_rate, 3),
} for m in (xgb_m, news_m)])

print(f"AUPRC 기준선(양성 비율) = {base:.4f}\n")
display(res)

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

xgb_s = model.predict_proba(split.X_test)[:, 1]
news_s = compute_news_scores(split.X_test)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, s, c in [("XGBoost", xgb_s, "C0"), ("NEWS", news_s, "C1")]:
    p, r, _ = precision_recall_curve(split.y_test, s); ax[0].plot(r, p, label=name, color=c, lw=2)
    f, t, _ = roc_curve(split.y_test, s);              ax[1].plot(f, t, label=name, color=c, lw=2)
ax[0].axhline(base, color="red", ls="--", label=f"baseline {base:.3f}")
ax[0].set_title("PR curve — where alarm quality shows"); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].legend()
ax[1].plot([0,1], [0,1], "k--", alpha=.3)
ax[1].set_title("ROC curve — often looks similar"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend()
plt.tight_layout(); plt.show()

두 그림을 비교해서 보세요. 이 프로젝트의 핵심 주장이 여기 있습니다.

ROC 곡선에서는 두 방법이 비슷해 보이는데 PR 곡선에서는 크게 벌어지는 경우가 많습니다.
드문 사건에서는 음성이 압도적으로 많아 ROC가 관대해지기 때문입니다.
임상에서 문제가 되는 건 "알람이 울렸을 때 진짜일 확률"(정밀도) 이고, 그건 PR 곡선에만 보입니다.

## 7. 알람 부담과 lead-time — 이 프로젝트가 진짜 보는 지표

### 알람 부담 (alarm burden)

> 같은 검출률(민감도)을 낼 때 알람을 몇 번 울리는가 — 낮을수록 좋음

두 방법의 민감도를 똑같이 맞춘 뒤 알람 횟수를 비교해야 공정합니다.
간호사가 실제로 체감하는 건 AUPRC가 아니라 "오늘 알람이 몇 번 울렸나" 니까요.

### lead-time

> 사건 몇 시간 전에 경보했는가 — 길수록 개입할 여유가 생김

너무 짧으면 (예: 10분 전) 경보해도 손쓸 수 없습니다.

In [ ]:
from vitals_train import alarm_burden, lead_time_summary, threshold_at_specificity

print("=== 동일 90% 민감도에서의 알람 부담 ===")
burden = {}
for name, s in [("XGBoost", xgb_s), ("NEWS", news_s)]:
    b = alarm_burden(split.y_test, s, target_sensitivity=0.90)
    burden[name] = b
    print(f"  {name:8s} 특이도={b['specificity']:.3f}  오경보율={b['false_alarm_rate']:.3f}  "
          f"알람수/100={b['alarms_per_100_windows']:.1f}")

lead = lead_time_summary(split, xgb_s, threshold_at_specificity(split.y_test, xgb_s))
if lead:
    print(f"\n=== 조기경보 확보시간 ===")
    print(f"  {int(lead['detected'])}/{int(lead['arrest_patients'])}명 검출, "
          f"중앙값 {lead['median_lead_time_h']:.1f}시간 전 (95% 특이도 기준)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

names = list(burden)
vals = [burden[n]["alarms_per_100_windows"] for n in names]
bars = ax[0].bar(names, vals, color=["seagreen", "indianred"])
ax[0].set_ylabel("alarms per 100 windows")
ax[0].set_title("Alarm burden at matched 90% sensitivity (lower = better)")
for b, v in zip(bars, vals):
    ax[0].text(b.get_x() + b.get_width()/2, v, f"{v:.1f}", ha="center", va="bottom")

# 민감도를 바꿔가며 알람 부담이 어떻게 변하는지
sens_grid = [0.70, 0.80, 0.90, 0.95, 0.99]
for name, s, c in [("XGBoost", xgb_s, "C0"), ("NEWS", news_s, "C1")]:
    y = [alarm_burden(split.y_test, s, target_sensitivity=t)["alarms_per_100_windows"]
         for t in sens_grid]
    ax[1].plot([t * 100 for t in sens_grid], y, marker="o", label=name, color=c, lw=2)
ax[1].set_xlabel("target sensitivity (%)"); ax[1].set_ylabel("alarms per 100 windows")
ax[1].set_title("Alarm burden across operating points"); ax[1].legend()
plt.tight_layout(); plt.show()

오른쪽 그래프가 이 프로젝트의 논지를 가장 잘 보여줍니다.

민감도를 높일수록(놓치는 환자를 줄일수록) 알람이 늘어납니다. 문제는 얼마나 가파르게 느는가예요.
NEWS는 고민감도 구간에서 알람이 폭증하는 반면, 좋은 모델은 완만하게 오릅니다.

"놓치지 않으면서도 알람을 적게" — 이 구간의 차이가 임상 도입 여부를 가릅니다.

## 8. SHAP — 왜 위험한지 설명

성능이 좋아도 근거를 못 대면 임상에서 안 씁니다. "이 환자가 위험합니다"만으로는
의료진이 무엇을 해야 할지 모르니까요.

SHAP은 각 피처가 예측을 위험/안전 쪽으로 얼마나 밀었는지 계산합니다.

In [ ]:
try:
    import shap
    sample = split.X_test.sample(min(500, len(split.X_test)), random_state=42)
    sv = shap.TreeExplainer(model).shap_values(sample.fillna(split.X_train.median()))
    if isinstance(sv, list):
        sv = sv[1]
    shap.summary_plot(sv, sample, max_display=15, show=False, plot_size=(10, 6))
    plt.title("SHAP — what drives the alarm")
    plt.tight_layout(); plt.show()

    imp = pd.Series(np.abs(sv).mean(axis=0), index=sample.columns).sort_values(ascending=False)
    print("영향력 상위 10개 피처:")
    print(imp.head(10).round(4).to_string())
except ImportError:
    print("shap 미설치 — pip install shap")

읽는 법: 세로축은 피처(위일수록 영향 큼), 가로축은 예측을 민 방향(오른쪽=위험),
색은 그 피처의 값(빨강=높음).

`_slope`(기울기) 피처가 상위에 오면 우리 설계가 맞았다는 뜻입니다 — "지금 값"보다
"변화하는 방향과 속도" 가 악화의 신호라는 가설이었으니까요.

반대로 의학 상식과 어긋나는 결과가 나오면 데이터를 의심해야 합니다.
SHAP은 성능 지표가 못 잡는 오류를 잡아냅니다.

## 9. 정리

### 시계열 조기경보의 흐름

```
긴 형태 원본 (시각, 항목, 값)
   ↓  어댑터            환자 × 시간 격자로 변환, 결측 처리
   ↓  EDA               사건 전에 신호가 실제로 있는지 확인
   ↓  슬라이딩 윈도우    시계열 → 학습 가능한 샘플 (지평 선택이 핵심)
   ↓  환자 단위 분할     누수 방지
   ↓  개인 기저선 피처   차별점
   ↓  XGBoost vs NEWS   임상 점수를 이기는가
   ↓  알람부담·lead-time 실제로 쓸 수 있는가
   ↓  SHAP              왜 위험한가
```

### 02번에 없던, 여기서 새로 배운 것

| 개념 | 요점 |
|---|---|
| 슬라이딩 윈도우 | 시계열을 잘라 각 조각을 샘플로. 기울기 피처가 중요 |
| 예측 지평 | 좁으면 양성이 사라져 AUPRC 붕괴. 임상 대응 시간으로 결정 |
| 환자 단위 분할 | 무작위 분할은 같은 환자가 양쪽에 들어가 성능을 부풀림 |
| 개인 기저선 | 병동 공통 임계값 대신 환자 자신 대비 편차 |
| 알람 부담 | 같은 민감도에서의 알람 횟수 — 임상이 체감하는 지표 |
| lead-time | 개입할 시간을 확보했는가 |

### 02번에서 배운 것도 그대로 적용됩니다

- AUPRC는 기준선(양성 비율)과 비교해서 읽기
- 정확도는 불균형에서 무의미
- 차이가 오차범위 안이면 "차이 없음"
- 피처 추가·앙상블이 항상 도움이 되는 건 아님

---

### 다음에 해볼 것

- `HORIZON`을 3, 12로 바꿔서 지표가 어떻게 변하는지
- `MAX_FILES=None`으로 4,000명 전체 실행 → 표본이 늘면 결론이 바뀌는지
- `WINDOW`를 4, 16으로 바꿔보기
- CLI로 튜닝까지: `python src/mortality_explore.py <폴더> --horizon=6 --tune --gpu`

### 실데이터 받는 법

```bash
cd /workspace
wget -r -N -c -np -nH --cut-dirs=4 \
  https://physionet.org/files/challenge-2012/1.0.0/set-a/
wget https://physionet.org/files/challenge-2012/1.0.0/Outcomes-a.txt
```

인증이 필요 없는 공개 데이터입니다 (ODC-BY). 받은 뒤 맨 위 `DATA_DIR`·`OUTCOMES`를
그 경로로 바꾸고 다시 실행하세요.